In [1]:
import os
import pandas as pd
import json
import pandas as pd
import numpy as np
import nltk
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import faiss
from geopy.distance import geodesic 
import pgeocode

# Check if the file exists
directory = r'C:\\Users\\hanjh\Downloads\\yelp_dataset'
os.chdir(directory)
print(os.listdir(directory))

# file_path = 'C:\\Users\\hanjh\\Downloads\\yelp_dataset'
grubhub = pd.read_csv('grubhub.csv') #pd.read_csv('grubhub_w_zip.csv') 
doordash = pd.read_csv('doordash.csv')
uber_rest = pd.read_csv('uber_restaurants.csv')
uber_menu = pd.read_csv('uber_restaurant_menus.csv')

['380K_US_Restaurants.csv', 'Dataset_User_Agreement.pdf', 'doordash.csv', 'grubhub.csv', 'grubhub_w_zip.csv', 'Gruphub.txt', 'uber_restaurants.csv', 'uber_restaurant_menus.csv', 'yelp_academic_dataset_business.json', 'yelp_academic_dataset_checkin.json', 'yelp_academic_dataset_review.json', 'yelp_academic_dataset_tip.json', 'yelp_academic_dataset_user.json']


In [ ]:
# 1. Grubhub (Note: Retrieving the zip code for each restaurant based on lat/long will take a while)

import time
pd.set_option('display.max_columns', None)  
grubhub= grubhub[['loc_name', 'cuisines','latitude','longitude','address']]
grubhub['cuisines'] = grubhub['cuisines'].apply(lambda x: ', '.join([item.strip().strip('"') for item in x[1:-1].split(',')]))
grubhub.loc[:, 'cleaned_text'] = grubhub['cuisines'].str.cat(grubhub['loc_name'], sep=' ')

from geopy.geocoders import Nominatim
from geopy.exc import GeocoderTimedOut, GeocoderServiceError

# Function to get zip code from latitude and longitude
def get_zip_code(lat, lon):
    geolocator = Nominatim(user_agent="check_1")
    try:
        location = geolocator.reverse((lat, lon), exactly_one=True)
        if location and 'postcode' in location.raw['address']:
            return location.raw['address']['postcode']
        else:
            return None
    except Exception as e:
        print(f"Error fetching zip code for coordinates ({lat}, {lon}): {e}")
        return None

# Apply the function to get zip codes for each restaurant
grubhub['zipcode'] = grubhub.apply(lambda row: get_zip_code(row['latitude'], row['longitude']), axis=1)

# Display the DataFrame with zip codes
grubhub.head()

# Please save it to a CSV file with a zip code column to avoid having to run the code again! 
# grubhub.to_csv('grubhub_w_zip.csv', index=False)

In [2]:
# 2.doordash
import re
doordash = pd.read_csv('doordash.csv')
doordash= doordash[['loc_name', 'cuisines','latitude','longitude','address']]

# Function to extract ZIP code before ', USA'
def extract_zipcode(address):
    match = re.search(r'(\d{5})(?=,\s*USA$)', address) # Match 5-digit ZIP code followed by ', USA'
    return match.group(1) if match else None  # Return only the ZIP code part

doordash['zipcode'] = doordash['address'].apply(extract_zipcode)
doordash['cuisines'] = doordash['cuisines'].str.replace('|', ', ')
doordash.loc[:, 'cleaned_text'] = doordash['cuisines'].str.cat(doordash['loc_name'], sep=' ')
doordash.head()


,loc_name,cuisines,latitude,longitude,address,zipcode,cleaned_text
0,Lotus Vietnamese Sandwiches,"Vietnamese, Bubble Tea, Smoothies, Sandwiches,...",40.675464,-73.980782,"229 5th Avenue, Brooklyn, NY 11215, USA",11215,"Vietnamese, Bubble Tea, Smoothies, Sandwiches,..."
1,Lotus Vietnamese Sandwiches,"Vietnamese, Bubble Tea, Smoothies, Sandwiches,...",40.675464,-73.980782,"229 5th Avenue, Brooklyn, NY 11215, USA",11215,"Vietnamese, Bubble Tea, Smoothies, Sandwiches,..."
2,Lotus Vietnamese Sandwiches,"Vietnamese, Bubble Tea, Smoothies, Sandwiches,...",40.675464,-73.980782,"229 5th Avenue, Brooklyn, NY 11215, USA",11215,"Vietnamese, Bubble Tea, Smoothies, Sandwiches,..."
3,Taqueria Milear,"Mexican, Tacos, Burritos, Dessert, Brunch",40.672978,-73.950462,"752 Nostrand Ave, Brooklyn, NY 11216, USA",11216,"Mexican, Tacos, Burritos, Dessert, Brunch Taqu..."
4,Taqueria Milear,"Mexican, Tacos, Burritos, Dessert, Brunch",40.672978,-73.950462,"752 Nostrand Ave, Brooklyn, NY 11216, USA",11216,"Mexican, Tacos, Burritos, Dessert, Brunch Taqu..."


In [3]:
# Uber
uber_rest = pd.read_csv('uber_restaurants.csv')
uber_rest= uber_rest[['id', 'name','category','full_address','zip_code','lat','lng']]

In [4]:
import re
def extract_five_digit_zipcode(zipcode):
    if isinstance(zipcode, str) or isinstance(zipcode, (int, float)):
        match = re.search(r'^\d{5}', str(zipcode))  # Match 5 digits at the start
        return match.group(0) if match else None  # Return the matched ZIP code
    return None  

# Apply the function to the zip_code column
uber_rest['zip_code'] = uber_rest['zip_code'].apply(extract_five_digit_zipcode)

def extract_name(name):
    match = re.match(r'([^(]+)', name)  # Match everything before the first parenthesis
    return match.group(1).strip() if match else name 

# Apply the function to the name column
uber_rest['cleaned_name'] = uber_rest['name'].apply(extract_name)

uber_menu = pd.read_csv('uber_restaurant_menus.csv')
uber_menu['category'] = uber_menu['category'].str.strip() # Strip whitespace from all entries in the 'category' column
uber_menu = uber_menu[['restaurant_id', 'category']].drop_duplicates() # Select specific columns and drop duplicates
uber_menu = uber_menu.groupby('restaurant_id', as_index=False).agg({
    'category': ', '.join 
})

In [5]:
uber_comb = uber_rest.merge(uber_menu, left_on='id', right_on='restaurant_id', how='left')
uber_comb['category'] = uber_comb['category_x'].fillna('') + (', ' if (uber_comb['category_x'].notna() & uber_comb['category_y'].notna()).any() else '') + uber_comb['category_y'].fillna('')

# Strip any leading or trailing whitespace, and remove any leading commas
uber_comb['category'] = uber_comb['category'].str.strip().str.lstrip(',')

uber_comb.loc[:, 'cleaned_text'] = uber_comb['category'].str.cat(uber_comb['name'], sep=' ')

In [6]:
df1= grubhub[['loc_name', 'cleaned_text', 'zipcode', 'address', 'cuisines','latitude','longitude']].rename(columns={'cuisines':'categories'})
df2= doordash[['loc_name', 'cleaned_text', 'zipcode','address', 'cuisines','latitude','longitude']].rename(columns={'cuisines':'categories'})
df3= uber_comb[['cleaned_name', 'cleaned_text', 'zip_code','full_address','category','lat','lng']].rename(columns={'cleaned_name':'loc_name', 'zip_code':'zipcode',
                                                                                                       'full_address':'address', 'category':'categories',
                                                                                                       'lat':'latitude','lng':'longitude'})
restaurant_df = pd.concat([df1,df2,df3]).rename(columns={'name':'cleaned_name', 'postal_code':'zip_code'})
restaurant_df= restaurant_df.rename(columns={'loc_name':'name', 'zipcode':'postal_code'})
restaurant_df.head()

,name,cleaned_text,postal_code,address,categories,latitude,longitude
0,Chun Vegetarian,"an, Vegan, Vegetar Chun Vegetarian",11216.0,582 Nostrand Ave,"an, Vegan, Vegetar",40.678829,-73.949867
1,India House,"lthy, Indian, Lunch Specials, Vegetar India House",11216.0,586 Nostand Ave,"lthy, Indian, Lunch Specials, Vegetar",40.678715,-73.949875
2,Pattie Hut,"cken, Soup, Wi Pattie Hut",11216.0,543 Nostrand Ave.,"cken, Soup, Wi",40.678879,-73.949448
3,Texas Chicken and Burgers,"cken, Hamburgers, Southern, Wi Texas Chicken a...",11216.0,551 Nostrand Ave,"cken, Hamburgers, Southern, Wi",40.678638,-73.949463
4,Grilled & Cheesy,"rican, Sandwic Grilled & Cheesy",11216.0,1306 Atlantic Ave,"rican, Sandwic",40.678226,-73.949440


In [7]:
file_path = 'C:\\Users\\hanjh\\Downloads\\yelp_dataset\\yelp_academic_dataset_business.json'

# Load the JSON data line by line
data = []
with open(file_path, 'r') as file:
    for line in file:
        data.append(json.loads(line))

# Access the data
print(data[0])  
print(f"Total entries: {len(data)}")  

{'business_id': 'Pns2l4eNsfO8kk83dixA6A', 'name': 'Abby Rappoport, LAC, CMQ', 'address': '1616 Chapala St, Ste 2', 'city': 'Santa Barbara', 'state': 'CA', 'postal_code': '93101', 'latitude': 34.4266787, 'longitude': -119.7111968, 'stars': 5.0, 'review_count': 7, 'is_open': 0, 'attributes': {'ByAppointmentOnly': 'True'}, 'categories': 'Doctors, Traditional Chinese Medicine, Naturopathic/Holistic, Acupuncture, Health & Medical, Nutritionists', 'hours': None}
Total entries: 150346


In [8]:
df = pd.DataFrame(data)
df['categories'] = df['categories'].astype(str)
df['name'] = df['name'].astype(str)
food_keywords = 'restaurant|food|dining|pizza|sushi|coffee|BBQ|bubble|vegan|Italian|Chinese|Thai|burger|bistro|steakhouse|takeout|brunch'
# Filter the DataFrame to keep rows where the 'categories' or 'name' contain the keywords
df = df[df['categories'].str.contains(food_keywords, case=False, na=False) | df['name'].str.contains(food_keywords, case=False, na=False)]

df['cleaned_text'] = df['categories'].str.cat(df['name'], sep=' ')
# Drop rows with NaN in 'cleaned_text'
df = df.dropna(subset=['cleaned_text'])

In [9]:
df=pd.concat([df[['name','cleaned_text','postal_code','address','categories','latitude','longitude']], restaurant_df])
df_clean = df.dropna().drop_duplicates().reset_index(drop=True)
df_clean['postal_code'] = df_clean['postal_code'].str.strip()
df_clean.head()

,name,cleaned_text,postal_code,address,categories,latitude,longitude
0,"Abby Rappoport, LAC, CMQ","Doctors, Traditional Chinese Medicine, Naturop...",93101,"1616 Chapala St, Ste 2","Doctors, Traditional Chinese Medicine, Naturop...",34.426679,-119.711197
1,St Honore Pastries,"Restaurants, Food, Bubble Tea, Coffee & Tea, B...",19107,935 Race St,"Restaurants, Food, Bubble Tea, Coffee & Tea, B...",39.955505,-75.155564
2,Perkiomen Valley Brewery,"Brewpubs, Breweries, Food Perkiomen Valley Bre...",18054,101 Walnut St,"Brewpubs, Breweries, Food",40.338183,-75.471659
3,Sonic Drive-In,"Burgers, Fast Food, Sandwiches, Food, Ice Crea...",37015,615 S Main St,"Burgers, Fast Food, Sandwiches, Food, Ice Crea...",36.269593,-87.058943
4,Tsevi's Pub And Grill,"Pubs, Restaurants, Italian, Bars, American (Tr...",63123,8025 Mackenzie Rd,"Pubs, Restaurants, Italian, Bars, American (Tr...",38.565165,-90.321087


In [10]:
df_clean.shape
df_clean.isna().sum()

name               0
cleaned_text       0
postal_code     7463
address            0
categories         0
latitude           0
longitude          0
dtype: int64

In [11]:
import torch
import torch.nn.functional as F
from collections import Counter
import numpy as np
from math import log
from collections import Counter
from math import log
from scipy.sparse import csr_matrix

# Tokenize the documents and build vocabulary
def tokenize(doc):
    return doc.lower().split()

tokenized_docs = [tokenize(doc) for doc in df_clean['cleaned_text']]

# Build a vocabulary and limit its size
max_features = 8000  # Limit vocabulary to the top N features
tokenized_docs = [tokenize(doc) for doc in df_clean['cleaned_text']]
vocabulary_counter = Counter(word for doc in tokenized_docs for word in doc)
most_common_words = vocabulary_counter.most_common(max_features)
vocab_to_idx = {word: idx for idx, (word, _) in enumerate(most_common_words)}

# Calculate Term Frequencies (TF)
def term_frequency(doc, vocab_to_idx):
    tf = np.zeros(len(vocab_to_idx), dtype=np.float32)
    counter = Counter(doc)
    for word, count in counter.items():
        if word in vocab_to_idx:
            tf[vocab_to_idx[word]] = count
    return tf

# Calculate Inverse Document Frequency (IDF)
def inverse_document_frequency(tokenized_docs, vocab_to_idx):
    num_docs = len(tokenized_docs)
    idf = np.zeros(len(vocab_to_idx), dtype=np.float32)

    for idx, word in enumerate(vocab_to_idx):
        doc_count = sum(1 for doc in tokenized_docs if word in doc)
        idf[idx] = log((num_docs + 1) / (doc_count + 1)) + 1  # Smoothing

    return idf

tf_matrix = np.stack([term_frequency(doc, vocab_to_idx) for doc in tokenized_docs])
idf = inverse_document_frequency(tokenized_docs, vocab_to_idx)

tfidf_matrix = tf_matrix * idf  # Compute TF-IDF matrix
tfidf_matrix = F.normalize(torch.tensor(tfidf_matrix), p=2, dim=1).numpy()  # Normalize
# L2 norm dividing each element of the vector by the square root of the sum of the squared elements in that vectors

#### TF
The Term Frequency of a word 𝑡 in a document 𝑑 is the number of times the word appears in the document, divided by the total number of words in the document.

$$
\text{TF}(t, d) = \frac{\text{Number of times term } t \text{ appears in document } d}{\text{Total number of terms in document } d}
$$

Where:
* t = a specific term (word)
* d = a specific document

#### IDF
The Inverse Document Frequency measures how important a term is across all documents in the corpus. It is calculated as the logarithm of the total number of documents 
𝑁, divided by the number of documents containing the term 𝑡 

$$
\text{IDF}(t, D) = \log\left(\frac{N}{n_t + 1}\right)
$$


#### TF-IDF
$$
\text{TF-IDF}(t, d, D) = \text{TF}(t, d) \times \text{IDF}(t, D)
$$

In [12]:
# Show the TF-IDF matrix
print("Vocabulary:", {k: vocab_to_idx[k] for k in list(vocab_to_idx)[:10]}) # only 10 index
print("Len of Vocabulary:", len(vocab_to_idx))

Vocabulary: {'food,': 0, '&': 1, 'for': 2, 'you,': 3, 'picked': 4, '&amp;': 5, 'restaurants,': 6, 'american,': 7, 'and': 8, 'sandwiches,': 9}
Len of Vocabulary: 8000


#### Search bar with zipcode

In [13]:
def search(query, postal_code=None, city=None, k=20, distance_threshold=1.8): # k : top 20 best results from search query / distance_threshold = 1.8 (tune)
    filtered_df = df_clean.copy()
        
    # Filter by postal code
    if postal_code:
        filtered_df = filtered_df[filtered_df['postal_code'] == postal_code]    
    
    if filtered_df.empty:
        return pd.DataFrame([])  
    
    # Transform the filtered businesses' cleaned_text to TF-IDF
    filtered_tokenized_docs = [tokenize(doc) for doc in filtered_df['cleaned_text']]
    filtered_tf_matrix = np.stack([term_frequency(doc, vocab_to_idx) for doc in filtered_tokenized_docs])
    filtered_tfidf_matrix = filtered_tf_matrix * idf[:filtered_tf_matrix.shape[1]]  # Compute TF-IDF for filtered documents
    filtered_tfidf_matrix = F.normalize(torch.tensor(filtered_tfidf_matrix), p=2, dim=1).numpy()  # Normalize

    # Build a temporary FAISS index for the filtered matrix
    temp_index = faiss.IndexFlatL2(filtered_tfidf_matrix.shape[1]) #L2 Euclidean distance, (Feedback: consider using Cosine similiary.)
    temp_index.add(filtered_tfidf_matrix.astype('float32'))

    # Transform the user input query to the TF-IDF space
    query_vector = term_frequency(tokenize(query), vocab_to_idx) * idf[:len(vocab_to_idx)]
    query_vector = F.normalize(torch.tensor(query_vector).unsqueeze(0), p=2, dim=1)  # Normalize

    # Convert the query vector to a NumPy array for FAISS
    query_vector_np = query_vector.detach().numpy().astype('float32')  # Ensure it's a NumPy array

    # Perform search on the FAISS index
    distances, indices = temp_index.search(query_vector_np, k)

    # Collect results
    results = []
    for i in range(k):
        if indices[0][i] < len(filtered_df) and distances[0][i] <= distance_threshold:
            business = filtered_df.iloc[indices[0][i]]
            results.append({
                'name': business['name'],
                'categories': business['categories'],
                'address': business['address'],
                'postal_code': business['postal_code'],
                'distance': distances[0][i]  
            })
    results_df = pd.DataFrame(results).reset_index(drop=True)
    
    pd.set_option('display.max_columns', None)  
    pd.set_option('display.expand_frame_repr', False)  

    return results_df

# Example search with postal code filtering
result = search('Chinese', postal_code='85746')
print(result)


                       name                                 categories                          address postal_code  distance
0   China Dragon Restaurant                       Restaurants, Chinese  1625 W Valencia Rd, Ste 101-103       85746  1.149650
1           Phoenix Village                       Restaurants, Chinese               2750 W Valencia Rd       85746  1.223154
2  China Olive Super Buffet  Buffets, Restaurants, Sushi Bars, Chinese     1350 W Irvington Rd, Ste 100       85746  1.401469


#### Search bar with zipcode and radius

In [14]:
# Code to retrieve lat/long from postal code to calculate miles.
import pgeocode

nomi = pgeocode.Nominatim('us')
query = nomi.query_postal_code("85746")

geo_info = {
    "lat": query["latitude"],
    "lon": query["longitude"]
}

print(geo_info)

{'lat': 32.1422, 'lon': -111.0506}


In [15]:
# Function to get latitude and longitude from postal code
def get_lat_lon(postal_code):
    query = nomi.query_postal_code(postal_code)
    if query is None:
        return None, None  # Handle case when postal code is not valid
    return query['latitude'], query['longitude']

# Haversine distance calculation (in miles)
def haversine_distance(lat1, lon1, lat2, lon2):
    return geodesic((lat1, lon1), (lat2, lon2)).miles

def search(query, postal_code=None, radius_miles=20, k= 150, distance_threshold=1.8):
    # Get user's latitude and longitude from the postal code
    if postal_code:
        user_lat, user_lon = get_lat_lon(postal_code)
        if user_lat is None or user_lon is None:
            return pd.DataFrame([])  # Return empty if postal code is invalid
    else:
        return pd.DataFrame([])  
    
     # Calculate the distance from the user location to each restaurant
    df_clean['distance_miles'] = df_clean.apply(lambda row: haversine_distance(user_lat, user_lon, row['latitude'], row['longitude']), axis=1)

    # Filter restaurant within the specified radius
    filtered_df = df_clean[df_clean['distance_miles'] <= radius_miles]
    # print(f"Filtered restaurants within {radius_miles} miles: {filtered_df.shape[0]} found.")

    if filtered_df.empty:
        return pd.DataFrame([])   

     # Transform the filtered restaurant cleaned_text to TF-IDF
    filtered_tokenized_docs = [tokenize(doc) for doc in filtered_df['cleaned_text']]
    filtered_tf_matrix = np.stack([term_frequency(doc, vocab_to_idx) for doc in filtered_tokenized_docs])
    filtered_tfidf_matrix = filtered_tf_matrix * idf[:filtered_tf_matrix.shape[1]]  # Compute TF-IDF for filtered documents
    filtered_tfidf_matrix = F.normalize(torch.tensor(filtered_tfidf_matrix), p=2, dim=1).numpy()  # Normalize

    # Build a temporary FAISS index for the filtered restaurant
    temp_index = faiss.IndexFlatL2(filtered_tfidf_matrix.shape[1])
    temp_index.add(filtered_tfidf_matrix.astype('float32'))

    # Transform the user input query to the TF-IDF space
    query_vector = term_frequency(tokenize(query), vocab_to_idx) * idf[:len(vocab_to_idx)]
    query_vector = F.normalize(torch.tensor(query_vector).unsqueeze(0), p=2, dim=1)  # Normalize

    # Convert the query vector to a NumPy array for FAISS
    query_vector_np = query_vector.detach().numpy().astype('float32')  # Ensure it's a NumPy array

    # Perform search on the FAISS index
    distances, indices = temp_index.search(query_vector_np, k)
    # Debug: Print the indices and distances retrieved
    # print("Indices and distances from FAISS search:")
    # for i in range(k):
    #     print(f"Index: {indices[0][i]}, Distance: {distances[0][i]}")

    # Collect results
    results = []
    for i in range(k):
        if indices[0][i] < len(filtered_df) and distances[0][i] <= distance_threshold:
            business = filtered_df.iloc[indices[0][i]]
            results.append({
                'Index': indices[0][i],  # Add the index for reference
                'Name': business['name'],
                'Categories': business['categories'],
                'Address': business['address'],
                'Postal Code': business['postal_code'],
                'Distance (miles)': business['distance_miles'],
                'Distance Score': distances[0][i]  # optional: show the distance score
            })
        # else:
        #     # Print info about skipped restaurants
        #     print(f"Skipped Index: {indices[0][i]}, Distance: {distances[0][i]}")

    # Create a DataFrame and reset index
    results_df = pd.DataFrame(results).reset_index(drop=True)
    
    pd.set_option('display.max_columns', None)  
    pd.set_option('display.expand_frame_repr', False)  

    return results_df

# Example search with postal code and radius in miles
result = search('Chinese', postal_code='85746', radius_miles=5)
print(result)

   Index                       Name                                 Categories                          Address Postal Code  Distance (miles)  Distance Score
0    101             House of Cheng                       Restaurants, Chinese           5975 W Western Way Cir       85713          3.857869        0.845943
1    143       China Bay Restaurant                       Restaurants, Chinese                 65 W Valencia Rd       85706          4.824009        1.133033
2      0    China Dragon Restaurant                       Restaurants, Chinese  1625 W Valencia Rd, Ste 101-103       85746          3.044757        1.149650
3      1  La Bella China Restaurant                       Restaurants, Chinese                  5680 S 12th Ave       85706          4.274521        1.208368
4    116            Phoenix Village                       Restaurants, Chinese               2750 W Valencia Rd       85746          1.537671        1.223154
5     10   China Olive Super Buffet  Buffets, Restau

In [16]:
def evaluate_precision_recall(query, postal_code=None, radius_miles=5, distance_threshold=1.8):

    retrieved_results = search(query, postal_code, radius_miles, distance_threshold=distance_threshold)
    print("Retrieved Results:", retrieved_results)  # Print the retrieved results for debugging

    # normalize column names to lowercase
    retrieved_results.columns = retrieved_results.columns.str.lower()

    # Define ground truth (all relevant restaurants with the query keyword in categories)
    ground_truth = df[
        df['cleaned_text'].str.contains(query, case=False, na=False) & 
        (df['postal_code'] == postal_code)  # Ensure it matches the given postal code
    ]

    print("Ground Truth:", ground_truth)  # Print the ground truth for debugging

    # normalize ground truth column names to lowercase
    ground_truth.columns = ground_truth.columns.str.lower()

    # Count true positives (correctly retrieved items)
    true_positives = retrieved_results[retrieved_results['name'].isin(ground_truth['name'])]

    # Calculate precision and recall
    precision = len(true_positives) / len(retrieved_results) if len(retrieved_results) > 0 else 0
    recall = len(true_positives) / len(ground_truth) if len(ground_truth) > 0 else 0  # Avoid division by zero

    return precision, recall

# Example usage with a simplified query
precision, recall = evaluate_precision_recall('Chinese', postal_code='85746', radius_miles=5)
print(f'Precision: {precision:.2f}, Recall: {recall:.2f}')

Retrieved Results:    Index                       Name                                 Categories                          Address Postal Code  Distance (miles)  Distance Score
0    101             House of Cheng                       Restaurants, Chinese           5975 W Western Way Cir       85713          3.857869        0.845943
1    143       China Bay Restaurant                       Restaurants, Chinese                 65 W Valencia Rd       85706          4.824009        1.133033
2      0    China Dragon Restaurant                       Restaurants, Chinese  1625 W Valencia Rd, Ste 101-103       85746          3.044757        1.149650
3      1  La Bella China Restaurant                       Restaurants, Chinese                  5680 S 12th Ave       85706          4.274521        1.208368
4    116            Phoenix Village                       Restaurants, Chinese               2750 W Valencia Rd       85746          1.537671        1.223154
5     10   China Olive Super Buff

#### Sementaic Search

In [17]:
from sentence_transformers import SentenceTransformer
import numpy as np

# Load a pre-trained model for sentence embeddings
model = SentenceTransformer('all-MiniLM-L6-v2')

document_embeddings = model.encode(df_clean['cleaned_text'].tolist(), show_progress_bar=True, convert_to_numpy=True)

def encode_query(query):
    return model.encode(query, convert_to_numpy=True)

c:\Users\hanjh\anaconda3\Lib\site-packages\sentence_transformers\cross_encoder\CrossEncoder.py:13: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange


Batches:   0%|          | 0/4347 [00:00<?, ?it/s]

In [21]:
print("Shape:", document_embeddings.shape)
print("Number of Dimensions:", document_embeddings.ndim)
print("First Element:", document_embeddings[0, 0])  # Accesses the element in the first row, first column
print("All Rows, First Column:", document_embeddings[:, 0])  # Accesses all rows in the first column

Shape: (139091, 384)
Number of Dimensions: 2
First Element: 0.02077506
All Rows, First Column: [ 0.02077506 -0.00023083  0.02074491 ...  0.00724849 -0.04855026
 -0.07108673]


In [22]:
import faiss

# Create a FAISS index
index = faiss.IndexFlatL2(document_embeddings.shape[1])  # L2 distance
index.add(document_embeddings)

In [24]:
df_clean.head()

,name,cleaned_text,postal_code,address,categories,latitude,longitude,distance_miles
0,"Abby Rappoport, LAC, CMQ","Doctors, Traditional Chinese Medicine, Naturop...",93101,"1616 Chapala St, Ste 2","Doctors, Traditional Chinese Medicine, Naturop...",34.426679,-119.711197,525.224617
1,St Honore Pastries,"Restaurants, Food, Bubble Tea, Coffee & Tea, B...",19107,935 Race St,"Restaurants, Food, Bubble Tea, Coffee & Tea, B...",39.955505,-75.155564,2065.527512
2,Perkiomen Valley Brewery,"Brewpubs, Breweries, Food Perkiomen Valley Bre...",18054,101 Walnut St,"Brewpubs, Breweries, Food",40.338183,-75.471659,2050.746919
3,Sonic Drive-In,"Burgers, Fast Food, Sandwiches, Food, Ice Crea...",37015,615 S Main St,"Burgers, Fast Food, Sandwiches, Food, Ice Crea...",36.269593,-87.058943,1399.219565
4,Tsevi's Pub And Grill,"Pubs, Restaurants, Italian, Bars, American (Tr...",63123,8025 Mackenzie Rd,"Pubs, Restaurants, Italian, Bars, American (Tr...",38.565165,-90.321087,1248.250030


In [25]:
def semantic_search(query, postal_code=None, radius_miles=5, k=20):
    # Get user's latitude and longitude from postal code
    if postal_code:
        user_lat, user_lon = get_lat_lon(postal_code)
        print(f"User Latitude: {user_lat}, User Longitude: {user_lon}")  # Check user location
    else:
        return pd.DataFrame([])  # Return empty if no postal code

    # Filter based on geographic proximity
    df_clean['distance_miles'] = df_clean.apply(lambda row: haversine_distance(user_lat, user_lon, row['latitude'], row['longitude']), axis=1)

    # Debugging step: Check if distance_miles is being created
    # print(df_clean[['name', 'distance_miles']].head())  # Print distances for first few rows

    # Filter restaurants based on the radius
    filtered_df = df_clean[df_clean['distance_miles'] <= radius_miles].reset_index(drop=True)

    # If no restaurants are found within the radius
    if filtered_df.empty:
        print("No restaurants found within the specified radius.")  # Informative message
        return pd.DataFrame([])

    # Get the embeddings for the filtered restaurants
    mask = df_clean['distance_miles'] <= radius_miles
    filtered_embeddings = document_embeddings[mask.values]  # Use boolean mask

    # Encode the query
    query_embedding = encode_query(query)

    # Search for the nearest neighbors
    distances, indices = index.search(np.array([query_embedding]), k)

    # Prepare the results
    results = []
    for i in range(k):
        if indices[0][i] < len(filtered_df):
            business = filtered_df.iloc[indices[0][i]]
            results.append({
                'name': business['name'],
                'categories': business['categories'],
                'address': business['address'],
                'postal_code': business['postal_code'],
                'distance_miles': business['distance_miles'],
                'distance_score': distances[0][i]
            })

    return pd.DataFrame(results)

# Example usage
result = semantic_search('Chinese', postal_code='85746', radius_miles=5)
print(result)

User Latitude: 32.1422, User Longitude: -111.0506
                       name  distance_miles
0  Abby Rappoport, LAC, CMQ      525.224617
1        St Honore Pastries     2065.527512
2  Perkiomen Valley Brewery     2050.746919
3            Sonic Drive-In     1399.219565
4     Tsevi's Pub And Grill     1248.250030
Empty DataFrame
Columns: []
Index: []
